# Monocle3 — *C. elegans* embryo (Python port)

Python port of the R vignette `c_elegans_embryo_v2.ipynb`. This notebook walks through the full Monocle3 trajectory analysis using `monocle3-python`, the AnnData-based reimplementation of the R package.

**Scope**: data loading, size factor estimation, PCA, batch / covariate alignment, UMAP, clustering, principal graph learning, pseudotime ordering, trajectory-based differential expression, gene modules, and 3D trajectory visualisation.

The dataset is the Packer et al. 2019 *C. elegans* embryo scRNA-seq atlas (6188 cells × 20222 genes).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

import monocle3 as m3
print('monocle3-python', m3.__version__)

## 2. Load worm embryo data (Packer et al. 2019)

The loader pulls `packer_embryo.h5ad` from cwd-local staging, the `~/.cache/monocle3-python/` cache, or downloads it from Zenodo on first use. `AnnData` replaces the R `cell_data_set` throughout the port.

In [ ]:
adata = m3.load_packer_embryo()
adata

## 3. Preprocess and align on loading covariates + batch

`preprocess_cds` runs size-factor normalisation, log-transform, and truncated PCA (matching R's `irlba::prcomp_irlba`). `align_cds` then subtracts linear effects of loading covariates and MNN-aligns on `batch`.

In [ ]:
m3.preprocess_cds(adata, num_dim=50)
m3.align_cds(
    adata,
    residual_model_formula_str=(
        '~ bg.300.loading + bg.400.loading + bg.500.1.loading + '
        'bg.500.2.loading + bg.r17.loading + bg.b01.loading + bg.b02.loading'
    ),
    alignment_group='batch',
)

## 4. UMAP coloured by annotated `cell.type`

`reduce_dimension` uses `umap-learn` with the same defaults as the R `uwot` call. `fast_sgd=False, cores=1` makes the embedding deterministic.

In [ ]:
np.random.seed(42)
m3.reduce_dimension(adata, umap_fast_sgd=False, cores=1)
m3.plot_cells(adata, label_groups_by_cluster=False, color_cells_by='cell.type')

## 5. Cluster cells (single partition at this resolution)

In [ ]:
m3.cluster_cells(adata, resolution=5e-6)
m3.plot_cells(adata, group_cells_by='partition', color_cells_by='partition')

## 6. Learn principal graph

The default `ncenter` heuristic produces too coarse a graph for embryo data. Setting `ncenter=1000` matches the R vignette.

In [ ]:
m3.learn_graph(adata, learn_graph_control={'ncenter': 1000})

m3.plot_cells(
    adata, color_cells_by='cell.type',
    label_groups_by_cluster=False,
    label_leaves=False, label_branch_points=False,
)

In [ ]:
m3.plot_cells(
    adata, color_cells_by='embryo.time.bin',
    label_cell_groups=False,
    label_leaves=True, label_branch_points=True,
    graph_label_size=1.5,
)

## 7. Order cells along pseudotime using a time-bin-aware root

Pseudotime zero is placed at the principal-graph node whose neighbourhood contains the most cells from the earliest time bin (`130-170`).

In [ ]:
def earliest_principal_node(adata, time_bin='130-170'):
    mask = adata.obs['embryo.time.bin'].astype(str) == time_bin
    closest = adata.uns['monocle3']['principal_graph_aux']['UMAP'][
        'pr_graph_cell_proj_closest_vertex'
    ]['V1'].to_numpy()
    closest_subset = closest[mask.to_numpy()]
    values, counts = np.unique(closest_subset, return_counts=True)
    return [f'Y_{int(values[np.argmax(counts)])}']

m3.order_cells(adata, root_pr_nodes=earliest_principal_node(adata))
m3.plot_cells(
    adata, color_cells_by='pseudotime',
    label_cell_groups=False,
    label_leaves=False, label_branch_points=False,
    graph_label_size=1.5,
)

## 8. Ciliated-neuron marker panel on the trajectory

In [ ]:
ciliated_genes = ['che-1', 'hlh-17', 'nhr-6', 'dmd-6', 'ceh-36', 'ham-1']

m3.plot_cells(
    adata, genes=ciliated_genes,
    label_cell_groups=False, show_trajectory_graph=False,
)

In [ ]:
gene_mask = adata.var['gene_short_name'].astype(str).isin(ciliated_genes)
m3.plot_percent_cells_positive(
    adata[:, gene_mask].copy(),
    group_cells_by='time.point',
)

## 9. Differential expression against `embryo.time`

In [ ]:
adata.obs['pseudotime'] = m3.pseudotime(adata)
sub = adata[:, gene_mask].copy()

gene_fits_time = m3.fit_models(sub, model_formula_str='~embryo.time')
fit_coefs = m3.coefficient_table(gene_fits_time)

(
    fit_coefs
    .query("term in ('embryo.time', 'embryo_time')")
    .query('q_value < 0.05')
    [['gene_short_name', 'term', 'q_value', 'estimate']]
)

In [ ]:
violin_mask = adata.var['gene_short_name'].astype(str).isin(
    ['che-1', 'nhr-6', 'dmd-6', 'ceh-36']
)
m3.plot_genes_violin(
    adata[:, violin_mask].copy(),
    group_cells_by='embryo.time.bin', ncol=2,
)

## 10. Adding `batch` as a covariate + fit evaluation

In [ ]:
gene_fits_batch = m3.fit_models(sub, model_formula_str='~embryo.time + batch')

m3.coefficient_table(gene_fits_batch) \
    .query("term not in ('Intercept', '(Intercept)')") \
    [['gene_short_name', 'term', 'q_value', 'estimate']]

In [ ]:
m3.evaluate_fits(gene_fits_batch)

## 11. LR test: does `batch` actually help? (negbinomial)

In [ ]:
time_models = m3.fit_models(
    sub, model_formula_str='~embryo.time', expression_family='negbinomial'
)
time_batch_models = m3.fit_models(
    sub, model_formula_str='~embryo.time + batch', expression_family='negbinomial'
)

m3.compare_models(time_batch_models, time_models)[['gene_id', 'q_value']]

## 12. DE along pseudotime via natural spline

In [ ]:
finite = np.isfinite(adata.obs['pseudotime'])
sub_pt = adata[finite.to_numpy() & gene_mask.reindex(adata.var_names).to_numpy(), :]
# Python: subset cells first; genes already filtered by gene_mask.
sub_pt = adata[finite.to_numpy()][:, gene_mask].copy()

gene_fits_pt = m3.fit_models(
    sub_pt,
    model_formula_str='~splines::ns(pseudotime, df=3)',
)
(
    m3.coefficient_table(gene_fits_pt)
    .loc[lambda d: d['term'].str.contains('pseudotime')]
    .assign(q_value=lambda d: d['q_value'].fillna(1.0))
    .query('q_value < 0.05')
    [['gene_short_name', 'term', 'q_value', 'estimate']]
)

## 13. Principal-graph-wide DE — Moran's I + gene modules

In [ ]:
pr_graph_test_res = m3.graph_test(adata, neighbor_graph='principal_graph', cores=4)
pr_deg_ids = pr_graph_test_res.query('q_value < 0.05').index.tolist()

gene_module_df = m3.find_gene_modules(adata[:, pr_deg_ids].copy(), resolution=1e-3)
cell_group_df = pd.DataFrame({
    'cell': adata.obs_names,
    'cell_group': adata.obs['cell.type'].astype(str).to_numpy(),
})

agg_mat = m3.aggregate_gene_expression(
    adata,
    gene_group_df=gene_module_df[['id', 'module']],
    cell_group_df=cell_group_df,
)
agg_mat.index = ['Module ' + str(i) for i in agg_mat.index]

import pheatmap
pheatmap.pheatmap(
    agg_mat, scale='column', clustering_method='ward.D2', fontsize=6,
)

In [ ]:
m3.plot_cells(adata, genes=gene_module_df, label_cell_groups=False, show_trajectory_graph=False)

## 14. AFD-lineage gene dynamics along pseudotime

The upstream tutorial hard-codes cluster IDs 22 / 28 / 35 for the AFD lineage. Our clustering won't match those IDs, so we pick the three most populous clusters.

In [ ]:
cluster_counts = m3.clusters(adata).value_counts()
top_clusters = cluster_counts.head(3).index.tolist()
afd_mask = m3.clusters(adata).isin(top_clusters)

afd_cds = adata[afd_mask.to_numpy()].copy()
afd_genes_mask = afd_cds.var['gene_short_name'].astype(str).isin(
    ['gcy-8', 'dac-1', 'oig-8']
)

m3.plot_genes_in_pseudotime(
    afd_cds[:, afd_genes_mask].copy(),
    color_cells_by='embryo.time.bin',
    min_expr=0.5,
)

## 15. 3D trajectory (re-embed with `max_components = 3`)

`plot_cells_3d` is the only site where `plotly.py` is imported in the whole port.

In [ ]:
cds_3d = m3.load_packer_embryo()
m3.preprocess_cds(cds_3d, num_dim=50)
m3.align_cds(cds_3d, alignment_group='batch')
m3.reduce_dimension(cds_3d, max_components=3)
m3.cluster_cells(cds_3d)
m3.learn_graph(cds_3d)
m3.order_cells(cds_3d, root_pr_nodes=earliest_principal_node(cds_3d))

m3.plot_cells_3d(cds_3d, color_cells_by='partition')